### New and hopefully better summarizer

In [1]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACTING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
openai = OpenAI()

In [3]:
# Class to represent a webpage
class Website:
    """
    Utility class to represent a website that has been scraped for data
    """
    url: str
    title: str
    text: str
    
    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content)
        self.title = soup.title.string if soup.title else "Title not found"
        
        for irrelevant in soup.body(["script", "style","img","input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:
website = Website("https://doom.bethesda.net/en-US/the-dark-ages")

In [5]:
website.title
website.text

"Pre-Order\nDOOM: The Dark Ages\nPre-Order Now\nMAY 15, 2025\nNews\nPrepare for DOOM: The Dark Ages with our PC specs\nJanuary 23\nDOOM: The Dark Ages Preorder, Premium Edition & Collector’s Bundle Details\nJanuary 23\nNew Gameplay Details Revealed for DOOM: The Dark Ages, Launching May 15, 2025\nJanuary 23\nSee DOOM: The Dark Ages at Developer_Direct ‘25\nJanuary 23\nDOOM: The Dark Ages revealed\nJune 09\nBECOME THE SLAYER IN A MEDIEVAL WAR AGAINST HELL\nDOOM: The Dark Ages is the prequel to the critically acclaimed DOOM (2016) and DOOM Eternal that tells an epic cinematic story of the DOOM Slayer's rage. In this third installment of the modern DOOM series, players will step into the blood-stained boots of the DOOM Slayer, in this never-before-seen dark and sinister medieval war against Hell.\nA DOOM FOR ALL SLAYERS\nA dark fantasy/sci-fi single-player experience that delivers the searing combat and over-the-top visuals of the incomparable DOOM franchise, powered by the latest idTech 

In [6]:
system_prompt = "You are an assistant that analyzes the contesnts of a website and provides a short summary, ignoring the text that might be navigation related. Respond in markdown"

In [22]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}. "
    user_prompt += "The contents of this website is as follows; please provide a short summary of this website in markdown.\
        if it indicates news or announcements, summarize those too\n\n"
    user_prompt += website.text
    return user_prompt
    

In [23]:
user_prompt_for(website)

"You are looking at a website titled DOOM: The Dark Ages. The contents of this website is as follows; please provide a short summary of this website in markdown.        if it indicates news or announcements, summarize those too\n\nPre-Order\nDOOM: The Dark Ages\nPre-Order Now\nMAY 15, 2025\nNews\nPrepare for DOOM: The Dark Ages with our PC specs\nJanuary 23\nDOOM: The Dark Ages Preorder, Premium Edition & Collector’s Bundle Details\nJanuary 23\nNew Gameplay Details Revealed for DOOM: The Dark Ages, Launching May 15, 2025\nJanuary 23\nSee DOOM: The Dark Ages at Developer_Direct ‘25\nJanuary 23\nDOOM: The Dark Ages revealed\nJune 09\nBECOME THE SLAYER IN A MEDIEVAL WAR AGAINST HELL\nDOOM: The Dark Ages is the prequel to the critically acclaimed DOOM (2016) and DOOM Eternal that tells an epic cinematic story of the DOOM Slayer's rage. In this third installment of the modern DOOM series, players will step into the blood-stained boots of the DOOM Slayer, in this never-before-seen dark and s

In [24]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)},
    ]

In [25]:
messages_for(website)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contesnts of a website and provides a short summary, ignoring the text that might be navigation related. Respond in markdown'},
 {'role': 'user',
  'content': "You are looking at a website titled DOOM: The Dark Ages. The contents of this website is as follows; please provide a short summary of this website in markdown.        if it indicates news or announcements, summarize those too\n\nPre-Order\nDOOM: The Dark Ages\nPre-Order Now\nMAY 15, 2025\nNews\nPrepare for DOOM: The Dark Ages with our PC specs\nJanuary 23\nDOOM: The Dark Ages Preorder, Premium Edition & Collector’s Bundle Details\nJanuary 23\nNew Gameplay Details Revealed for DOOM: The Dark Ages, Launching May 15, 2025\nJanuary 23\nSee DOOM: The Dark Ages at Developer_Direct ‘25\nJanuary 23\nDOOM: The Dark Ages revealed\nJune 09\nBECOME THE SLAYER IN A MEDIEVAL WAR AGAINST HELL\nDOOM: The Dark Ages is the prequel to the critically acclaimed DOOM (2016) and

In [28]:
def summarizer(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [29]:
summarizer("https://doom.bethesda.net/en-US/the-dark-ages")

"# DOOM: The Dark Ages Summary\n\n**Overview**  \nDOOM: The Dark Ages is an upcoming game set to launch on May 15, 2025. It serves as a prequel to the critically acclaimed *DOOM (2016)* and *DOOM Eternal*, focusing on the epic and cinematic story of the DOOM Slayer's rage in a dark, medieval war against Hell. The game promises a dark fantasy/sci-fi single-player experience that retains the signature intense combat and impressive visuals of the DOOM franchise, powered by the latest idTech engine.\n\n**Gameplay Features**  \nPlayers will step into the role of the DOOM Slayer, utilizing a range of powerful weapons, including the Super Shotgun and a new tool called the Shield Saw, to combat demonic forces. The game will feature customizable difficulty to cater to both new players and veterans, alongside expansive levels filled with challenges and mysteries set in various dark realms.\n\n**Recent News and Announcements**  \n- **January 23, 2025**: Announced gameplay details and specificatio

In [30]:
display(Markdown(summarizer("https://doom.bethesda.net/en-US/the-dark-ages")))

# Summary of DOOM: The Dark Ages Website

**Release Date:** May 15, 2025

DOOM: The Dark Ages is a prequel to the acclaimed DOOM (2016) and DOOM Eternal, featuring an epic cinematic story that delves into the rage of the DOOM Slayer amidst a dark, medieval war against Hell. This title aims to provide both new players and long-time fans with a compelling single-player experience that includes customizable difficulty settings and the signature visceral combat of the DOOM franchise, powered by the latest idTech engine.

## Key Features:
- **Dark Fantasy/Sci-Fi Experience:** Engage in brutal combat using a variety of weapons, including the Super Shotgun and the new Shield Saw.
- **Cinematic Story:** Follow the Slayer as he battles against demon hordes led by an antagonist striving to become the only feared force.
- **Expansive World:** Explore unknown realms with richly designed environments such as ruined castles, dark forests, and hellscapes.

## News and Announcements:
- **PC Specs:** Detailed specifications were released for optimal gameplay.
- **Preorder Details:** Information on the Premium Edition and Collector’s Bundle became available.
- **Gameplay Reveal:** New gameplay mechanics were showcased, highlighting the game’s features.
- **Developer Insight:** The upcoming title was revealed at Developer_Direct ‘25.

Players are encouraged to join the Slayers Club for updates and community engagement.

### Now Add Selenium for JS rendered webpages